# 15. Dropout — борьба с переобучением

Видео: [15. Dropout. Борьба с переобучением](https://www.youtube.com/watch?v=42xf6g4yPIA&list=PLBP4Q3FNSLK2rtGPBsK-aMAetYj-8yg_1&index=16) (~5 мин)

## Зачем этот урок

Train loss падает, val растёт / расходится → сеть **зубрит**. Уменьшить сеть — ок, но часто режет точность. **Dropout** оставляет размер сети, но на каждом батче случайно гасит нейроны → остальные учатся подстраховывать.

> **Конспект:** [`d.4.3`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.3 Регуляризация в Deep Learning/d.4.3 Регуляризация в Deep Learning.md) **§2.2** (+ пример [`d.4.3.3`](../../../все мои конспекты в ynote/d.Машинное обучение/d.4 Deep Learning/d.4.3 Регуляризация в Deep Learning/d.4.3.3 Пример — Dropout и чем он отличается от ReLU.md)). Карта — [`0_как_учить_pytorch.md`](../0_как_учить_pytorch.md).

| блок | тема |
|:---|:---|
| 1 | `nn.Dropout(p)` после ReLU |
| 2 | `train()` включает / `eval()` выключает |
| 3 | почему на val с Dropout часто «лучше», чем на train |
| 4 | куда ставить в `nn.Module` (как у автора) |

**Как смотреть.** Целиком (~5 мин): идея → после активации → train/eval → коллаб с MNIST.

**Как решать.** Замени `...` → проверка. Решения — `15_dropout_решения.ipynb`.


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)


def need(cond, msg):
    assert cond, msg


print("torch", torch.__version__)


## 1. <span style="color:#7CDBA0">`nn.Dropout(p)`</span> — после активации

**Как у автора:** сигнал доходит до слоя → **часть нейронов гасится** с вероятностью `p` → дальше идут только «выжившие». На следующем батче маска **новая**.

```python
nn.Linear(...) → nn.ReLU() → nn.Dropout(p=0.5) → nn.Linear(...)
```

В PyTorch уже **inverted dropout**: выжившие × `1/(1-p)` на train, на eval — identity (масштаб совпадает).


In [ ]:
# 1.1 слой Dropout
# p — вероятность ВЫКЛЮЧИТЬ нейрон (не «оставить»)
drop = nn.Dropout(p=0.5)
print(drop)
need(isinstance(drop, nn.Dropout), "nn.Dropout")
need(abs(drop.p - 0.5) < 1e-9, "p=0.5")
print("1.1 — верно")


### 1.2 Крошечная сеть с Dropout

Как у автора на MNIST: `Dropout` **сразу после** ReLU, перед следующим `Linear`.


In [ ]:
# 1.2 MLP с Dropout после ReLU
class TinyDrop(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.Linear_1 = nn.Linear(4, 8)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(p)
        self.Linear_2 = nn.Linear(8, 2)

    def forward(self, x):
        x = self.Linear_1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.Linear_2(x)
        return x


torch.manual_seed(0)
net = TinyDrop(p=0.5)
x = torch.randn(16, 4)
y = net(x)
need(tuple(y.shape) == (16, 2), f"out shape {tuple(y.shape)}")
need(isinstance(net.drop, nn.Dropout), "self.drop = nn.Dropout(...)")
print("1.2 — верно", tuple(y.shape))


## 2. <span style="color:#7CDBA0">`train` / `eval`</span> — когда Dropout жив

| режим | Dropout |
|:---|:---|
| `model.train()` | **включён** — случайные нули |
| `model.eval()` | **выключен** — все нейроны «на поле» |

Без `eval()` на val/test предсказания будут **шумными** (и каждый прогон другой).  
`torch.no_grad()` **не** выключает Dropout — только градиенты.


In [ ]:
# 2.1 один вход: train шумит, eval стабилен
torch.manual_seed(0)
net = TinyDrop(p=0.5)
xb = torch.randn(32, 4)

net.train()
torch.manual_seed(1)
y_tr_a = net(xb).detach().clone()
torch.manual_seed(2)
y_tr_b = net(xb).detach().clone()

net.eval()
with torch.no_grad():
    y_ev_a = net(xb).detach().clone()
    y_ev_b = net(xb).detach().clone()

need(not torch.allclose(y_tr_a, y_tr_b), "в train два прогона с разным seed маски → разные выходы")
need(torch.allclose(y_ev_a, y_ev_b), "в eval два прогона → одинаково")
print("train diff:", (y_tr_a - y_tr_b).abs().mean().item())
print("eval diff:", (y_ev_a - y_ev_b).abs().mean().item())
print("2.1 — верно")


In [ ]:
# 2.2 eval: Dropout = identity; train: меняет тензор (гасит часть ненулевых после ReLU)
torch.manual_seed(0)
net = TinyDrop(p=0.5)
xb = torch.randn(64, 4)

h = net.act(net.Linear_1(xb))

net.eval()
h_ev = net.drop(h)
need(torch.allclose(h, h_ev), "в eval Dropout не меняет тензор")

net.train()
torch.manual_seed(7)
h_tr = net.drop(h)
need(not torch.allclose(h, h_tr), "в train Dropout меняет тензор")
killed = ((h_tr == 0) & (h != 0)).float().mean().item()
print(f"доля убитых Dropout-ом (из ненулевых ReLU): {killed:.2f}")
need(killed > 0.1, "в train Dropout гасит часть живых активаций")
print("2.2 — верно")


## 3. <span style="color:#7CDBA0">Почему val может быть «лучше» train</span>

У автора на MNIST с Dropout: **val loss / accuracy долго лучше train**. Это нормально:

- на **train** работает «дырявая» подсеть;
- на **val** (`eval`) — **вся** сеть целиком.

И как раз меньше классического расхождения «train улетел вниз, val стоит».  
Подбирай `p` (часто начинают с `0.5` на скрытом; слишком большой `p` → недообучение).


In [ ]:
# 3.1 sanity: p слишком большой → сильнее шум на train
torch.manual_seed(0)
mild = TinyDrop(p=0.2)
hard = TinyDrop(p=0.8)
xb = torch.randn(64, 4)

mild.train()
hard.train()
torch.manual_seed(3)
z_mild = mild.drop(mild.act(mild.Linear_1(xb)))
torch.manual_seed(3)
z_hard = hard.drop(hard.act(hard.Linear_1(xb)))
frac_mild = (z_mild == 0).float().mean().item()
frac_hard = (z_hard == 0).float().mean().item()
print(f"нулей p=0.2 → {frac_mild:.2f}; p=0.8 → {frac_hard:.2f}")
need(frac_hard > frac_mild, "больший p → больше нулей на train")
print("3.1 — верно")


## 4. <span style="color:#7CDBA0">Шаблон в классе</span> (как переносить в проект)

```python
class Net(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.Linear_1 = nn.Linear(784, 128)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(p)   # после активации
        self.Linear_2 = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.act(self.Linear_1(x))
        x = self.drop(x)
        return self.Linear_2(x)
```

В цикле как раньше: каждая эпоха `model.train()` на train-батчах, потом `model.eval()` + `no_grad` на val.

**Не путать с ReLU:** ReLU даёт 0 при z<0 (часть формулы нейрона, и на test тоже). Dropout на test **выключен** — см. `d.4.3.3`.


In [ ]:
# 4.1 Fashion-подобная голова с Dropout
class FashionLikeDrop(nn.Module):
    def __init__(self, hidden=64, p=0.5):
        super().__init__()
        self.flatten = nn.Flatten()
        self.Linear_1 = nn.Linear(28 * 28, hidden)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(p)
        self.Linear_2 = nn.Linear(hidden, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.Linear_1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.Linear_2(x)
        return x


torch.manual_seed(0)
m = FashionLikeDrop(p=0.5)
xb = torch.randn(8, 1, 28, 28)
m.train()
yt = m(xb)
m.eval()
with torch.no_grad():
    ye = m(xb)
need(tuple(yt.shape) == (8, 10), "train out")
need(tuple(ye.shape) == (8, 10), "eval out")
need("drop" in m.state_dict() or any("drop" in n for n, _ in m.named_modules()), "Dropout в модуле")
# Dropout без весов — в state_dict ключей drop.* нет; проверяем модуль:
need(any(isinstance(mod, nn.Dropout) for mod in m.modules()), "есть nn.Dropout в modules()")
print("4.1 — верно")


## Итог

| факт | помнить |
|:---|:---|
| `nn.Dropout(p)` | p = вероятность **выключить** |
| куда | обычно **после** ReLU (как у автора) |
| `train()` | Dropout **вкл** |
| `eval()` | Dropout **выкл** |
| val лучше train | часто **норма** с Dropout |
| ReLU ≠ Dropout | разные нули — см. `d.4.3.3` |

Дальше — **видео 16**: BatchNorm.  
Проект: раздел «Видео 15» в `project/fashion_mnist.ipynb`.
